# 01 - Data UnderstandingPresentation surface over `src/data/`. **No analysis logic lives in this notebook** -everything here calls the same modules `scripts/prepare_data.py` uses, so nothing showncan drift from what the pipeline actually does.Prerequisite:```bashpython scripts/prepare_data.py --tier dev```

In [ ]:
import sys, jsonfrom pathlib import PathROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(ROOT))import pandas as pdfrom src.utils.config import load_config, resolve_pathfrom src.utils.logging import setup_loggingsetup_logging("WARNING")          # keep the notebook readablepd.set_option("display.width", 160)pd.set_option("display.max_columns", 40)data_config = load_config("data_config")PROCESSED = resolve_path(data_config.require("paths.processed"))INTERIM   = resolve_path(data_config.require("paths.interim"))TABLES    = resolve_path(data_config.require("paths.tables"))FIGURES   = resolve_path(data_config.require("paths.figures"))print("tier:", data_config.require("active_tier"))

## 1. What the loader actually foundThe manifest records provenance, the sampling strategy, and its declared bias.

In [ ]:
manifest = json.loads((INTERIM / "load_manifest.json").read_text())print("tier      :", manifest["tier"], "|", manifest["category"])print("strategy  :", manifest["sampling"]["strategy"], "seed", manifest["sampling"]["seed"])print()print("BIAS DECLARATION")print(manifest["sampling"]["bias_declaration"])print()for name, source in manifest["sources"].items():    print(f"{name:8s} {source['bytes'] / 2**30:6.2f} GB  materialised={source['materialised']}")pd.Series(manifest["counts"]).to_frame("count")

## 2. Schema adaptationThe canonical column names used everywhere in `src/` versus the source field names.Fields listed as **declared absent** genuinely do not exist in Amazon Reviews 2023 -they are reported, never invented.

In [ ]:
availability = manifest["field_availability"]for entity in ("interactions", "products"):    info = availability[entity]    if info is None:        continue    print(f"--- {entity} ---")    display(pd.Series(info["mapped"], name="source_field").to_frame())    if info["declared_absent"]:        print("DECLARED ABSENT (not in the dataset):", info["declared_absent"])    if info["missing_in_data"]:        print("EXPECTED BUT NOT FOUND:", info["missing_in_data"])    print()

## 3. The cleaned data

In [ ]:
interactions = pd.read_parquet(PROCESSED / "interactions_clean.parquet")products     = pd.read_parquet(PROCESSED / "products_clean.parquet")print(f"interactions {interactions.shape}   products {products.shape}")print(f"users {interactions.user_id.nunique():,}  items {interactions.item_id.nunique():,}")print(f"span  {interactions.event_time.min().date()} -> {interactions.event_time.max().date()}")interactions.head(3)

### Invariants the rest of the project relies onIf any of these fail, a later phase is building on sand.

In [ ]:
k = data_config["tiers"][data_config.require("active_tier")]["k_core"]checks = {    "ratings within [1, 5]":        bool(interactions.rating.between(1, 5).all()),    "no duplicate (user, item)":    not interactions.duplicated(["user_id", "item_id"]).any(),    "sorted chronologically":       bool(interactions.timestamp.is_monotonic_increasing),    "no null user/item keys":       bool(interactions[["user_id", "item_id"]].notna().all().all()),    f"min user degree >= {k['min_interactions_per_user']}":        int(interactions.user_id.value_counts().min()) >= int(k["min_interactions_per_user"]),    f"min item degree >= {k['min_interactions_per_item']}":        int(interactions.item_id.value_counts().min()) >= int(k["min_interactions_per_item"]),}pd.Series(checks).to_frame("holds")

## 4. Cleaning reportEvery row removed, and why. Suspicious rows are **flagged and retained** - the anomaly detector cannot find what cleaning already deleted.

In [ ]:
from IPython.display import MarkdownMarkdown((TABLES / "cleaning_report.md").read_text())

In [ ]:
flagged = interactions[interactions.outlier_flag]print(f"{len(flagged):,} of {len(interactions):,} rows flagged ({100*len(flagged)/len(interactions):.1f}%), 0 removed")(flagged.outlier_reasons.str.split(";").explode().value_counts().to_frame("rows"))

## 5. Known gaps in this datasetCarried into `README.md` limitations and `docs/viva.md`.

In [ ]:
gaps = pd.DataFrame([    ("brand",   "ABSENT - `store` (storefront name) used as a proxy", "Brand nodes are approximate"),    ("seller",  "ABSENT entirely",                                    "No Seller node type exists"),    ("fraud labels", "ABSENT",                       "Trust is unsupervised; synthetic validation only"),    ("price",   f"null for {100*products.price_missing.mean():.1f}% of products", "Never imputed"),    ("categories", f"empty for {100*products.categories.apply(len).eq(0).mean():.1f}% of products",     "category_node falls back to main_category"),], columns=["field", "status", "consequence"])gaps